# Part 3 - Email: Outlook Support Emails, Reply Drafts, Sending and Customer Answers

## Goal

From the planning document:

- *"A request enters manually or through an authorized email thread."*
- *"A Communication Agent drafts clarification emails."*
- *"An email MCP can retrieve an authorized thread and save a reply as a draft, but sending requires human approval."*
- *"Unresolved questions are drafted and paused for human approval and answers."*

This notebook builds that loop on a real mailbox:

1. A **mail client** for Outlook through **Microsoft Graph** (or a local JSON mailbox when Graph is not
   configured). The assistant can only read **support emails**: emails from the team, internal senders,
   excluded domains and unrelated or confidential emails are filtered out (the same rules as the
   EmailToWorkItem app).
2. An **email MCP server** with five tools: list support emails, read an email, download its attachments,
   save a reply draft and send a draft.
3. An **approval tool** (ledger) that records human decisions. Saving a draft needs an approval of the email
   text for that message; sending needs a second approval for that exact draft. Auto mode never approves
   sending a real email.
4. The **Communication Agent**, which drafts the clarification email from the gaps, the company template and
   the communication standard.
5. An **answer-mapping agent**, which turns the customer's reply into confirmed answers and assumptions.

Notebook 04 uses the same email server to turn a support email into Azure DevOps work items and to send the
acknowledgement with the ticket number.

## Setup

Same environment as the course notebooks: a local **Ollama** server running **`gemma4:e4b`** for the agents and **`nomic-embed-text`** for embeddings. The cells work on Google Colab (Ollama is installed automatically) and locally (install Ollama from ollama.com first).

This notebook only needs the chat model.

**Chat model:** set `LLM_PROVIDER=openai` in `.env` to run the agents on **GPT Luna** (`OPENAI_MODEL`, default `gpt-6-luna`, key in `OPEN_AI_KEY`); Ollama is then not needed for this notebook. `LLM_PROVIDER=ollama` (the default) keeps the local `gemma4:e4b`.

In [ ]:
%pip install -q -U langchain langchain-core langchain-ollama langchain-openai python-dotenv msal langchain-text-splitters langgraph langchain-mcp-adapters mcp python-docx requests numpy langsmith

In [ ]:
import os
import platform
import shutil
import subprocess
import time

import requests
from dotenv import load_dotenv

load_dotenv()  # reads .env next to the notebook (OPEN_AI_KEY, LLM_PROVIDER, ADO_*, ...) when it exists

# Chat model: "openai" runs the agents on GPT Luna (key in OPEN_AI_KEY), "ollama" on the local model.
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama").strip().lower()
if LLM_PROVIDER not in ("openai", "ollama"):
    raise ValueError(f"LLM_PROVIDER must be 'openai' or 'ollama', not {LLM_PROVIDER!r}")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e4b")
LLM_MODEL = OPENAI_MODEL if LLM_PROVIDER == "openai" else OLLAMA_MODEL
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")
OLLAMA_URL = "http://localhost:11434"
NEEDS_OLLAMA = LLM_PROVIDER == "ollama"  # this notebook only needs the chat model


def ollama_is_running() -> bool:
    try:
        return requests.get(f"{OLLAMA_URL}/api/tags", timeout=3).ok
    except requests.RequestException:
        return False


# 1. Install Ollama (Colab / Linux) and start the server in the background
if NEEDS_OLLAMA:
    if shutil.which("ollama") is None:
        if platform.system() == "Linux":
            print("Installing Ollama...")
            !sudo apt-get install -y -qq zstd
            !curl -fsSL https://ollama.com/install.sh | sh
        else:
            print("Install Ollama from https://ollama.com/download, then re-run this cell.")

    if not ollama_is_running():
        print("Starting Ollama server...")
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(5)  # Wait for server to wake up

    print("Ollama running:", ollama_is_running())
print(f"Chat model: {LLM_PROVIDER} / {LLM_MODEL}")

In [ ]:
# 2. Pull the chat model and the embedding model (only when the agents run on Ollama)
if NEEDS_OLLAMA:
    !ollama pull {LLM_MODEL}
    !ollama pull {EMBED_MODEL}
    !ollama list

### LangSmith tracing

Every agent, LLM call, MCP tool call and LangGraph node is traced in **LangSmith**, so you can open each step
and see its **prompt, output, duration and tokens**. Paste your API key when asked (or store it as the Colab
secret `LANGSMITH_API_KEY`); press Enter to run without tracing.

In [ ]:
import getpass
import os

import langsmith.utils

# LangSmith records every agent, LLM call (full prompt and answer), MCP tool call and LangGraph node,
# with its duration and token usage. Free API key: https://smith.langchain.com -> Settings -> API Keys.
# Note: traces contain the prompts, so customer emails are sent to LangSmith. Use test data or your own workspace.
USE_LANGSMITH = True
LANGSMITH_PROJECT = "fo-requirements-to-delivery"


def read_langsmith_key() -> str:
    if os.getenv("LANGSMITH_API_KEY"):
        return os.environ["LANGSMITH_API_KEY"]
    try:
        from google.colab import userdata  # Colab: add a secret named LANGSMITH_API_KEY

        return userdata.get("LANGSMITH_API_KEY")
    except Exception:
        pass
    try:
        return getpass.getpass("LangSmith API key (press Enter to run without tracing): ").strip()
    except Exception:  # no interactive input available
        return ""


api_key = read_langsmith_key() if USE_LANGSMITH else ""
if api_key:
    os.environ.update(LANGSMITH_API_KEY=api_key, LANGSMITH_TRACING="true", LANGSMITH_PROJECT=LANGSMITH_PROJECT)
    print(f"LangSmith tracing ON -> project '{LANGSMITH_PROJECT}' on https://smith.langchain.com")
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing OFF")
langsmith.utils.get_env_var.cache_clear()  # LangSmith caches environment variables

In [ ]:
import getpass
import os


def make_llm():
    """The chat model selected by LLM_PROVIDER: GPT Luna on OpenAI, or the local Ollama model."""
    if LLM_PROVIDER == "openai":
        from langchain_openai import ChatOpenAI

        if not (os.getenv("OPEN_AI_KEY") or os.getenv("OPENAI_API_KEY")):
            os.environ["OPEN_AI_KEY"] = getpass.getpass("OpenAI API key: ").strip()
        # GPT Luna only accepts its default temperature, so none is set.
        return ChatOpenAI(model=LLM_MODEL, api_key=os.getenv("OPEN_AI_KEY") or os.getenv("OPENAI_API_KEY"))

    from langchain_ollama import ChatOllama

    return ChatOllama(model=LLM_MODEL, temperature=0)


llm = make_llm()

In [ ]:
import json

from langchain_core.messages import HumanMessage, SystemMessage
from langsmith import traceable  # names agent steps in LangSmith (does nothing when tracing is off)


def ask_structured(schema, system_prompt: str, user_prompt: str, retries: int = 2):
    """Call the LLM and force the answer into a Pydantic schema. On failure, retry with the error message."""
    # The run name shows up in LangSmith, e.g. "DeliveryPlan" or "GapAnalysis".
    structured_llm = llm.with_structured_output(schema, method="json_schema").with_config(run_name=schema.__name__)
    last_error = None

    for attempt in range(retries + 1):
        try:
            result = structured_llm.invoke(
                [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
            )
            if result is None:
                raise ValueError("The model returned an empty answer.")
            return result
        except Exception as error:
            last_error = error
            print(f"[retry {attempt + 1}] invalid structured output: {error}")
            user_prompt += (
                f"\n\nYour previous answer was invalid ({error}). "
                "Return JSON that matches the schema exactly."
            )

    raise last_error


def ask_text(system_prompt: str, user_prompt: str) -> str:
    return llm.invoke(
        [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
    ).text  # plain string for both providers (OpenAI returns content blocks)


def show(title: str, obj) -> None:
    """Pretty-print a heading and a Pydantic object, dict, list or string."""
    print(f"\n========== {title} ==========\n")
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    print(obj if isinstance(obj, str) else json.dumps(obj, indent=2, ensure_ascii=False))
from datetime import date, timedelta
from pathlib import Path

from pydantic import BaseModel, Field

In [ ]:
class ClarificationEmail(BaseModel):
    subject: str
    greeting: str = Field(description="Greeting line only, e.g. 'Dear Rana,'")
    introduction: str = Field(description="One-sentence restatement of the request")
    questions: list[str] = Field(description="Clarification questions, one topic each, without numbers")
    reply_by: str = Field(description="Only the requested reply date, e.g. '05 October 2026'")
    closing: str = Field(description="Closing phrase only, e.g. 'Kind regards,', without a name or signature")


class Answer(BaseModel):
    gap_id: str
    answer: str = Field(description="The customer's answer, with their exact values")


class CustomerAnswers(BaseModel):
    answers: list[Answer]
    unanswered_gap_ids: list[str]
    go_live_date: str = Field(default="", description="Go-live date as YYYY-MM-DD if the customer gave one, else empty")

In [ ]:
COMMUNICATION_SYSTEM = """
You are the Communication Agent of a Dynamics 365 F&O delivery team.
Draft a clarification email to the customer.

Follow the company template and communication standard you are given:
- Restate the request in one sentence, then ask the numbered questions (one topic each).
- Plain, polite business language. No technical jargon.
- Ask for a reply date. Never promise a delivery date or a solution.
"""

ANSWERS_SYSTEM = """
You map a customer's reply to open clarification questions (gaps).
- Record an answer only when the reply really answers that gap, and copy the customer's exact values.
- List every gap the reply does not answer in unanswered_gap_ids.
"""

## The Mailbox

The assistant works on a real Outlook mailbox through Microsoft Graph. Until Graph is configured, a **local
JSON mailbox** is used so the notebook runs anywhere (dry-run: nothing is read from or sent to a real mailbox):

- `MSG-1001`: the CR-2026-001 request used through this walkthrough,
- `MSG-1002`: a support issue with an error log attached (and an inline logo in the signature),
- `MSG-1003`: a confidential HR email, which the support filter must keep away from the assistant,
- `MSG-1004`: a posting error (a fix) together with a new field on a report (a change, so a Change Request).

## Connecting Outlook (Microsoft Graph)

Without Graph settings the local mailbox is used. To work on your real Outlook mailbox:

1. In the Azure portal, open **Microsoft Entra ID > App registrations > New registration** (single tenant).
2. Under **Authentication**, set **Allow public client flows** to **Yes** (needed for the device-code sign-in).
3. Under **API permissions**, add the Microsoft Graph **delegated** permissions `Mail.ReadWrite` and `Mail.Send`.
4. Put `GRAPH_CLIENT_ID` (Application ID) and `GRAPH_TENANT_ID` (Directory ID) in `.env` and re-run from here.
   The first run prints a link and a code: sign in once. The token is cached in `.graph_token_cache.json`
   (private, listed in `.gitignore`) and the MCP server reuses it.

For a shared support mailbox without a personal sign-in, use **application** permissions instead (admin
consent), add `GRAPH_CLIENT_SECRET` and `GRAPH_MAILBOX`, and restrict the app to that mailbox in Exchange Online.

The **support filter** comes from `.env`: `EMAIL_REQUIRED_KEYWORDS` (default `f&o,support`), `EXCLUDE_EMAILS`,
`TECHNICAL_PEOPLE`, `FUNCTIONAL_PEOPLE`, `DISINCLUDE_DOMAINS` and `EMAIL_INTERNAL_DOMAINS` (skipped unless
`EMAIL_INCLUDE_INTERNAL=true`). Replies to existing tickets (`... | 1234` in the subject) are skipped too.

In [ ]:
%%writefile mail_client.py
"""Mailbox client: Outlook through Microsoft Graph (live) or a local JSON mailbox (dry-run).

Live mode is used when GRAPH_CLIENT_ID and GRAPH_TENANT_ID are set (or GRAPH_ACCESS_TOKEN, for a quick test
with a token from Graph Explorer).
- Delegated (default): run MailClient().sign_in() once in the notebook. The token is cached in
  .graph_token_cache.json and refreshed silently, so the MCP server can use it without asking again.
- App-only: also set GRAPH_CLIENT_SECRET and GRAPH_MAILBOX (the shared support mailbox).

Only support emails can be read: get_email(), save_attachments() and create_reply_draft() refuse any message
that list_support_emails() would filter out (see support_filter).
"""
import base64
import html
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote

import requests

GRAPH_URL = os.getenv("GRAPH_BASE_URL", "https://graph.microsoft.com/v1.0")
SCOPES = ["Mail.ReadWrite", "Mail.Send"]
TOKEN_CACHE = Path(".graph_token_cache.json")
LOCAL_MAILBOX = Path("mailbox/messages.json")
OUTBOX = Path("mailbox/outbox.json")  # every draft and sent reply (both modes), for the audit trail
ATTACHMENTS_DIR = Path("attachments")
MAX_ATTACHMENT_BYTES = 25 * 1024 * 1024
OUTPUT_DIR = Path("output")  # generated documents: the only files that can be attached to a reply
MAX_SEND_ATTACHMENT_BYTES = 3 * 1024 * 1024
TICKET_SUBJECT = re.compile(r"\|\s*.*?\d+")  # replies to existing tickets, e.g. "Contoso - Fix posting | 1234"
MESSAGE_FIELDS = "id,conversationId,subject,from,toRecipients,ccRecipients,receivedDateTime,hasAttachments,body"


def _env_list(name: str, default: str = "") -> list:
    return [item.strip().lower() for item in os.getenv(name, default).split(",") if item.strip()]


def support_filter(message: dict) -> str:
    """Return why a message is not a support email for the assistant, or "" when it is."""
    subject, body = message["subject"].lower(), message["body"].lower()
    sender_name, sender = message["from_name"].lower(), message["from_address"].lower()
    staff = _env_list("TECHNICAL_PEOPLE") + _env_list("FUNCTIONAL_PEOPLE")

    if any(word in subject or word in sender_name for word in _env_list("EXCLUDE_EMAILS")):
        return "excluded keyword"
    if TICKET_SUBJECT.search(subject):
        return "reply to an existing ticket"
    if any(person in sender_name for person in staff):
        return "sent by a team member"
    if any(domain in sender for domain in _env_list("DISINCLUDE_DOMAINS")):
        return "excluded domain"
    include_internal = os.getenv("EMAIL_INCLUDE_INTERNAL", "false").strip().lower() in ("true", "1", "yes")
    if not include_internal and any(domain in sender for domain in _env_list("EMAIL_INTERNAL_DOMAINS")):
        return "internal sender"
    if not all(word in f"{subject} {body}" for word in _env_list("EMAIL_REQUIRED_KEYWORDS", "f&o,support")):
        return "not a support email"
    return ""


def to_html(text: str) -> str:
    """Plain text with **bold** markers -> HTML for the email body."""
    escaped = html.escape(text)
    return re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", escaped).replace("\n", "<br>\n")


def _safe_name(name: str) -> str:
    return re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", Path(name).name).strip() or "attachment"


def _now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


class MailClient:
    def __init__(self):
        self.client_id = os.getenv("GRAPH_CLIENT_ID", "")
        self.tenant_id = os.getenv("GRAPH_TENANT_ID", "")
        self.secret = os.getenv("GRAPH_CLIENT_SECRET", "")
        self.mailbox = os.getenv("GRAPH_MAILBOX", "")
        self.folder = os.getenv("GRAPH_MAIL_FOLDER", "inbox")
        self.static_token = os.getenv("GRAPH_ACCESS_TOKEN", "")
        self.live = bool(self.static_token or (self.client_id and self.tenant_id))
        if self.live and self.secret and not self.mailbox:
            raise ValueError("App-only Graph access needs GRAPH_MAILBOX (the mailbox to read and send from).")

    @property
    def mode(self) -> str:
        return "live" if self.live else "dry-run"

    # ---------------- Microsoft Graph authentication ----------------
    def _public_app(self):
        import msal

        cache = msal.SerializableTokenCache()
        if TOKEN_CACHE.exists():
            cache.deserialize(TOKEN_CACHE.read_text(encoding="utf-8"))
        app = msal.PublicClientApplication(
            self.client_id, authority=f"https://login.microsoftonline.com/{self.tenant_id}", token_cache=cache)
        return app, cache

    @staticmethod
    def _save_cache(cache) -> None:
        if cache.has_state_changed:
            TOKEN_CACHE.write_text(cache.serialize(), encoding="utf-8")

    def sign_in(self) -> str:
        """Delegated sign-in with a device code (open the link, type the code). Cached for later runs."""
        if not self.live or self.static_token or self.secret:
            return f"No interactive sign-in needed ({self.mode}{', app-only' if self.secret else ''})."
        app, cache = self._public_app()
        accounts = app.get_accounts()
        if accounts and app.acquire_token_silent(SCOPES, account=accounts[0]):
            self._save_cache(cache)
            return f"Already signed in as {accounts[0]['username']}"
        flow = app.initiate_device_flow(scopes=SCOPES)
        if "user_code" not in flow:
            raise RuntimeError(f"Could not start the device sign-in: {flow.get('error_description', flow)}")
        print(flow["message"], flush=True)
        result = app.acquire_token_by_device_flow(flow)
        if "access_token" not in result:
            raise RuntimeError(f"Sign-in failed: {result.get('error_description', result)}")
        self._save_cache(cache)
        return f"Signed in as {result.get('id_token_claims', {}).get('preferred_username', 'unknown user')}"

    def _token(self) -> str:
        if self.static_token:
            return self.static_token
        import msal

        if self.secret:
            app = msal.ConfidentialClientApplication(
                self.client_id, client_credential=self.secret,
                authority=f"https://login.microsoftonline.com/{self.tenant_id}")
            result = app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
        else:
            app, cache = self._public_app()
            accounts = app.get_accounts()
            result = app.acquire_token_silent(SCOPES, account=accounts[0]) if accounts else None
            if not result:
                raise PermissionError("Not signed in to Microsoft Graph: run MailClient().sign_in() in the notebook.")
            self._save_cache(cache)
        if "access_token" not in result:
            raise PermissionError(f"Could not get a Graph token: {result.get('error_description', result)}")
        return result["access_token"]

    def _graph(self, method: str, path: str, **kwargs) -> requests.Response:
        user = f"/users/{quote(self.mailbox)}" if self.secret else "/me"
        url = path if "://" in path else f"{GRAPH_URL}{user}{path}"  # @odata.nextLink is already absolute
        headers = {
            "Authorization": f"Bearer {self._token()}",
            # Immutable IDs stay valid when a message is moved to another folder.
            "Prefer": 'IdType="ImmutableId", outlook.body-content-type="text"',
            **kwargs.pop("headers", {}),
        }
        response = requests.request(method, url, headers=headers, timeout=60, **kwargs)
        if not response.ok:
            raise requests.HTTPError(f"Graph {method} {path} -> {response.status_code}: {response.text[:500]}",
                                     response=response)
        return response

    @staticmethod
    def _mid(message_id: str) -> str:
        return quote(message_id, safe="")

    # ---------------- reading ----------------
    @staticmethod
    def _from_graph(m: dict) -> dict:
        sender = (m.get("from") or {}).get("emailAddress", {})
        return {
            "message_id": m["id"],
            "conversation_id": m.get("conversationId", ""),
            "subject": m.get("subject") or "",
            "from_name": sender.get("name", ""),
            "from_address": sender.get("address", ""),
            "to": [r["emailAddress"]["address"] for r in m.get("toRecipients", [])],
            "cc": [r["emailAddress"]["address"] for r in m.get("ccRecipients", [])],
            "received": m.get("receivedDateTime", ""),
            "body": (m.get("body") or {}).get("content", ""),
            "has_attachments": bool(m.get("hasAttachments")),
        }

    def _local_messages(self) -> list:
        return json.loads(LOCAL_MAILBOX.read_text(encoding="utf-8")) if LOCAL_MAILBOX.exists() else []

    def _file_attachments(self, message: dict) -> list:
        """Real file attachments (inline images such as signature logos are skipped)."""
        if not self.live:
            items = [{**a, "id": a["name"]} for a in message.get("attachments", [])]
        elif not message.get("has_attachments"):
            return []
        else:
            items = self._graph(
                "GET", f"/messages/{self._mid(message['message_id'])}/attachments",
                params={"$select": "id,name,contentType,size,isInline"}).json()["value"]
            items = [{**a, "is_inline": a.get("isInline"), "content_type": a.get("contentType")} for a in items
                     if a.get("@odata.type") != "#microsoft.graph.referenceAttachment"]  # links have no file
        return [{"id": a["id"], "name": a["name"], "content_type": a.get("content_type", ""), "size": a.get("size", 0)}
                for a in items if not a.get("is_inline") and a.get("size", 0) <= MAX_ATTACHMENT_BYTES]

    def list_support_emails(self, max_emails: int = 10) -> list:
        """Newest support emails in the mailbox folder, after the support filter."""
        results = []
        if not self.live:
            pages = [sorted(self._local_messages(), key=lambda m: m["received"], reverse=True)]
        else:
            pages = self._graph_pages(max_emails)
        for page in pages:
            for message in page:
                if support_filter(message):
                    continue
                results.append({
                    "message_id": message["message_id"],
                    "subject": message["subject"],
                    "from": f"{message['from_name']} <{message['from_address']}>",
                    "received": message["received"],
                    "preview": " ".join(message["body"].split())[:300],
                    "attachments": [a["name"] for a in self._file_attachments(message)],
                })
                if len(results) >= max_emails:
                    return results
        return results

    def _graph_pages(self, max_emails: int):
        url = f"/mailFolders/{quote(self.folder)}/messages"
        params = {"$top": 50, "$orderby": "receivedDateTime desc", "$select": MESSAGE_FIELDS}
        scanned, limit = 0, max(200, max_emails * 50)  # guard against scanning a huge mailbox
        while url and scanned < limit:
            data = self._graph("GET", url, params=params).json()
            page = [self._from_graph(m) for m in data.get("value", [])]
            scanned += len(page)
            yield page
            url, params = data.get("@odata.nextLink"), None

    def _message(self, message_id: str) -> dict:
        if not self.live:
            for message in self._local_messages():
                if message["message_id"] == message_id:
                    return message
            raise ValueError(f"Unknown message: {message_id}")
        data = self._graph("GET", f"/messages/{self._mid(message_id)}", params={"$select": MESSAGE_FIELDS}).json()
        return self._from_graph(data)

    def _support_message(self, message_id: str) -> dict:
        message = self._message(message_id)
        reason = support_filter(message)
        if reason:
            raise PermissionError(f"Message {message_id} is not available to the assistant ({reason}).")
        return message

    def get_email(self, message_id: str) -> dict:
        """One support email with its full text and the names of its file attachments."""
        message = self._support_message(message_id)
        return {**{k: v for k, v in message.items() if k != "attachments"},
                "attachments": [{k: a[k] for k in ("name", "content_type", "size")}
                                for a in self._file_attachments(message)]}

    def save_attachments(self, message_id: str) -> list:
        """Download the file attachments of a support email to attachments/<message>/ and return their paths."""
        message = self._support_message(message_id)
        folder = ATTACHMENTS_DIR / re.sub(r"[^A-Za-z0-9]", "", message_id)[-16:]
        folder.mkdir(parents=True, exist_ok=True)
        saved = []
        for attachment in self._file_attachments(message):
            if self.live:
                content = self._graph("GET", f"/messages/{self._mid(message_id)}/attachments/"
                                             f"{self._mid(attachment['id'])}/$value").content
            else:
                stored = next(a for a in message["attachments"] if a["name"] == attachment["name"])
                content = base64.b64decode(stored["content_base64"])
            path = folder / _safe_name(attachment["name"])
            path.write_bytes(content)
            saved.append({"name": attachment["name"], "path": str(path.resolve()), "size": len(content),
                          "content_type": attachment["content_type"]})
        return saved

    # ---------------- drafting and sending ----------------
    def _outbox(self) -> list:
        return json.loads(OUTBOX.read_text(encoding="utf-8")) if OUTBOX.exists() else []

    def _save_outbox(self, records: list) -> None:
        OUTBOX.parent.mkdir(parents=True, exist_ok=True)
        OUTBOX.write_text(json.dumps(records, indent=2, ensure_ascii=False), encoding="utf-8")

    @staticmethod
    def _sendable(path: str) -> Path:
        """Only documents the notebooks generated in output/ may leave in a customer email."""
        file = Path(path).resolve()
        if not file.is_relative_to(OUTPUT_DIR.resolve()):
            raise PermissionError(f"Only documents generated in {OUTPUT_DIR}/ can be attached: {path}")
        if not file.is_file():
            raise FileNotFoundError(f"Attachment not found: {path}")
        if file.stat().st_size > MAX_SEND_ATTACHMENT_BYTES:
            raise ValueError(f"{file.name} is larger than 3 MB, the limit for a simple Graph attachment.")
        return file

    def create_reply_draft(self, message_id: str, subject: str, body: str, reply_all: bool = True,
                           attachment_paths: list | None = None) -> dict:
        """Create a reply (all) draft to a support email. Nothing is sent. Body: plain text, **bold** allowed.
        attachment_paths: generated documents from output/ (for example a Change Request) to attach."""
        message = self._support_message(message_id)
        files = [self._sendable(path) for path in attachment_paths or []]
        if self.live:
            action = "createReplyAll" if reply_all else "createReply"
            html_body = {"Prefer": 'IdType="ImmutableId"'}  # keep the quoted original as HTML
            draft = self._graph("POST", f"/messages/{self._mid(message_id)}/{action}", json={},
                                headers=html_body).json()
            quoted = (draft.get("body") or {}).get("content", "")
            new_body = to_html(body)
            match = re.search(r"<body[^>]*>", quoted, flags=re.IGNORECASE)
            content = (quoted[:match.end()] + new_body + "<br>" + quoted[match.end():]) if match \
                else new_body + "<br><hr>" + quoted
            draft = self._graph("PATCH", f"/messages/{self._mid(draft['id'])}", headers=html_body,
                                json={"subject": subject, "body": {"contentType": "HTML", "content": content}}).json()
            draft_id = draft["id"]
            for file in files:
                self._graph("POST", f"/messages/{self._mid(draft_id)}/attachments", json={
                    "@odata.type": "#microsoft.graph.fileAttachment", "name": file.name,
                    "contentBytes": base64.b64encode(file.read_bytes()).decode()})
            to = [r["emailAddress"]["address"] for r in draft.get("toRecipients", [])]
            cc = [r["emailAddress"]["address"] for r in draft.get("ccRecipients", [])]
        else:
            draft_id = f"DRAFT-{len(self._outbox()) + 1:03d}"
            to = [message["from_address"]]
            cc = list(message["cc"]) if reply_all else []  # like Graph: our own mailbox address is not copied

        record = {"draft_id": draft_id, "message_id": message_id, "mode": self.mode, "to": to, "cc": cc,
                  "subject": subject, "body": body, "attachments": [file.name for file in files],
                  "status": "draft - not sent", "created_at": _now()}
        self._save_outbox(self._outbox() + [record])
        return record

    def send_draft(self, draft_id: str) -> dict:
        """Send a draft created by create_reply_draft."""
        records = self._outbox()
        record = next((r for r in records if r["draft_id"] == draft_id), None)
        if record is None:
            raise ValueError(f"Unknown draft: {draft_id}")
        if record["status"].startswith("sent"):
            raise ValueError(f"Draft {draft_id} was already sent.")
        if self.live:
            self._graph("POST", f"/messages/{self._mid(draft_id)}/send")
            record["status"] = "sent"
        else:
            record["status"] = "sent (dry-run: nothing left this computer)"
        record["sent_at"] = _now()
        self._save_outbox(records)
        return record

In [ ]:
import importlib

import mail_client

importlib.reload(mail_client)
mail = mail_client.MailClient()
print("Mailbox mode:", mail.mode)
print(mail.sign_in())  # live + delegated: device-code sign-in the first time, cached afterwards

In [ ]:
import base64

REQUEST_ID = "CR-2026-001"

CUSTOMER_REQUEST = """
Subject: Vendor invoices approval + blocking vendors

Hi team,

Going forward we need vendor invoices above a certain amount to be approved by the
finance manager before they can be posted. Right now anyone in AP can post them and
the auditors flagged it.

Also, vendors that don't have bank details should be blocked so we don't pay them by
mistake. Some of the old vendors are like this too.

Can we have this for all our companies? We would like it live before the quarter close.

Thanks,
Rana Haddad
AP Team Lead, Cedar Retail Group
"""

SUPPORT_ISSUE = """
Hi support team,

Since this morning, sales orders in USMF cannot be confirmed. When we click "Confirm sales order"
we get the message "Update has been canceled" and the order stays open. This blocks today's shipments.

I attached the error log. Can you please check urgently?

Thanks,
Omar Saad
Sales Operations, Cedar Retail Group
"""

MIXED_REQUEST = """
Hello,

Since yesterday, posting free text invoices in USMF fails with the error "Number sequence FTI-USMF is exhausted"
(the error message is attached). We cannot invoice our customers until this is fixed.

Also, our auditors asked that the customer's VAT number be printed on the free text invoice report. Can you
add it?

Best regards,
Lina Saleh
Finance Manager, Northwind Traders
"""

ERROR_LOG = b"""2026-09-30 07:58:12 SalesConfirmJournalPost: Update has been canceled.
2026-09-30 07:58:12 Credit limit check: customer US-027 exceeds credit limit (52,300.00 USD).
"""
LOGO_PNG = base64.b64decode(
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mNk+M9QDwADhgGAWjR9awAAAABJRU5ErkJggg==")


def local_attachment(name: str, content: bytes, content_type: str, inline: bool = False) -> dict:
    return {"name": name, "content_type": content_type, "size": len(content), "is_inline": inline,
            "content_base64": base64.b64encode(content).decode()}


def local_message(message_id, name, address, subject, body, received, cc=(), attachments=()) -> dict:
    return {"message_id": message_id, "conversation_id": f"CONV-{message_id}", "subject": subject,
            "from_name": name, "from_address": address, "to": ["fo-support@partner.example"], "cc": list(cc),
            "received": received, "body": body.strip(), "has_attachments": bool(attachments),
            "attachments": list(attachments)}


LOCAL_MAILBOX = [
    local_message("MSG-1001", "Rana Haddad", "rana.haddad@cedarretail.example",
                  "F&O support - Vendor invoices approval + blocking vendors", CUSTOMER_REQUEST, "2026-09-25T09:14:00Z"),
    local_message("MSG-1002", "Omar Saad", "omar.saad@cedarretail.example",
                  "F&O support - Sales order confirmation fails", SUPPORT_ISSUE, "2026-09-30T08:02:00Z",
                  cc=["it.manager@cedarretail.example"],
                  attachments=[local_attachment("confirmation_error.log", ERROR_LOG, "text/plain"),
                               local_attachment("cedar_logo.png", LOGO_PNG, "image/png", inline=True)]),
    local_message("MSG-1003", "Cedar HR", "hr@cedarretail.example",
                  "Salary review - confidential", "Confidential HR content.", "2026-09-26T11:02:00Z"),
    local_message("MSG-1004", "Lina Saleh", "lina.saleh@northwindtraders.example",
                  "F&O support - Free text invoice posting error and VAT number on the report", MIXED_REQUEST,
                  "2026-10-01T10:30:00Z",
                  attachments=[local_attachment("posting_error.txt", b"Number sequence FTI-USMF is exhausted.", "text/plain")]),
]

Path("mailbox").mkdir(exist_ok=True)
Path("mailbox/messages.json").write_text(json.dumps(LOCAL_MAILBOX, indent=2), encoding="utf-8")
if not mail.live:
    Path("mailbox/outbox.json").unlink(missing_ok=True)  # the live outbox is the audit trail: never cleared
print("Local mailbox ready with", len(LOCAL_MAILBOX), "messages")

## Approval Tool

Every human decision is written to `approvals.json` with an approval ID. MCP tools check this ledger before
they act, so an agent cannot skip the human step by simply calling a tool. Email approvals are tied to the
exact message (email text) or draft (sending) they were given for.

In [ ]:
%%writefile approval_ledger.py
"""Approval tool: every human decision is recorded so that MCP tools can verify approval before acting."""
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path

LEDGER = Path("approvals.json")


def _load() -> list:
    return json.loads(LEDGER.read_text(encoding="utf-8")) if LEDGER.exists() else []


def record_decision(artefact_type: str, artefact_ref: str, approver: str, decision: str, comments: str = "") -> dict:
    record = {
        "approval_id": f"APR-{uuid.uuid4().hex[:8].upper()}",
        "artefact_type": artefact_type,
        "artefact_ref": artefact_ref,
        "approver": approver,
        "decision": decision,
        "comments": comments,
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    records = _load()
    records.append(record)
    LEDGER.write_text(json.dumps(records, indent=2), encoding="utf-8")
    return record


def is_approved(approval_id: str, artefact_type: str, artefact_ref: str | None = None) -> bool:
    """True when a person approved this artefact type (and, if given, this exact artefact)."""
    return any(
        r["approval_id"] == approval_id and r["artefact_type"] == artefact_type and r["decision"] == "approved"
        and (artefact_ref is None or r["artefact_ref"] == artefact_ref)
        for r in _load()
    )

In [ ]:
import importlib

import approval_ledger

importlib.reload(approval_ledger)
Path("approvals.json").unlink(missing_ok=True)

In [ ]:
# "auto" = scripted demo decisions (always approve, simulated customer reply)
# "interactive" = you type every decision and the customer's reply yourself
HUMAN_MODE = "auto"
APPROVER = "Toni Kanaan"


def human_decision(title: str, artefact: str, options=("approve", "revise", "reject"), live_action: bool = False) -> dict:
    """Show an artefact to a person and return their decision and comments.
    live_action: approving triggers a real action (a real email, real work items). Auto mode never approves it."""
    print(f"\n==================== HUMAN REVIEW: {title} ====================\n")
    print(artefact)
    print()

    if HUMAN_MODE == "interactive":
        decision = ""
        while decision not in options:
            decision = input(f"Decision {options}: ").strip().lower()
        comments = input("Comments (optional): ").strip()
    elif live_action:
        decision, comments = "reject", "Auto mode never approves a live action. Set HUMAN_MODE = 'interactive'."
        print(f"[auto] decision = {decision}: {comments}")
    else:
        decision, comments = "approve", "Approved in auto demo mode."
        print(f"[auto] decision = {decision}")

    return {"decision": decision, "comments": comments}

## Email MCP Server

| Tool | What it does | Guardrail |
|---|---|---|
| `list_support_emails` | Newest support emails: ID, subject, sender, preview, attachment names | Support filter |
| `get_email` | Full text, recipients and attachment names of one email | Refuses emails that fail the support filter |
| `save_attachments` | Downloads the file attachments to `attachments/` | Inline images skipped, 25 MB limit |
| `save_reply_draft` | Saves a reply-all as a draft in the mailbox (nothing is sent), optionally with generated documents from `output/` such as a Change Request | Approved `customer_email` for that message |
| `send_draft` | Sends a saved draft | Approved `send_email` for that exact draft |

In [ ]:
%%writefile email_server.py
"""Email MCP server: reads support emails from Outlook (Microsoft Graph) or the local mailbox, downloads their
attachments, saves reply drafts and sends them. Drafting and sending each need their own human approval."""
import json
import sys

from mcp.server.fastmcp import FastMCP

from approval_ledger import is_approved
from mail_client import MailClient

mcp = FastMCP("email")
mail = MailClient()


def log(message: str) -> None:
    print(f"[email tool:{mail.mode}] {message}", file=sys.stderr, flush=True)  # stdout is reserved for MCP


@mcp.tool()
def list_support_emails(max_emails: int = 10) -> str:
    """List the newest support emails the assistant may read (ID, subject, sender, preview, attachment names).
    Emails from the team, internal senders, excluded domains or confidential topics are filtered out."""
    log(f"list_support_emails(max_emails={max_emails})")
    return json.dumps(mail.list_support_emails(max_emails))


@mcp.tool()
def get_email(message_id: str) -> str:
    """Return one support email with its full text, recipients and attachment names."""
    log(f"get_email({message_id[:24]!r})")
    return json.dumps(mail.get_email(message_id))


@mcp.tool()
def save_attachments(message_id: str) -> str:
    """Download the file attachments of a support email (inline images are skipped) and return local paths."""
    log(f"save_attachments({message_id[:24]!r})")
    return json.dumps(mail.save_attachments(message_id))


@mcp.tool()
def save_reply_draft(message_id: str, subject: str, body: str, approval_id: str,
                     attachment_paths: list[str] | None = None) -> str:
    """Save a reply-all to a support email as a DRAFT in the mailbox. Nothing is sent.
    body: plain text, **bold** is allowed.
    attachment_paths: documents generated in output/ (for example a Change Request); other files are refused.
    approval_id: an approved 'customer_email' decision for this message_id from the approval ledger."""
    log(f"save_reply_draft(message_id={message_id[:24]!r}, approval_id={approval_id!r}, "
        f"attachments={len(attachment_paths or [])})")
    if not is_approved(approval_id, "customer_email", message_id):
        raise PermissionError("No human approval found for this email text. A person must approve it first.")
    return json.dumps(mail.create_reply_draft(message_id, subject, body, attachment_paths=attachment_paths))


@mcp.tool()
def send_draft(draft_id: str, approval_id: str) -> str:
    """Send a saved draft to the customer.
    approval_id: an approved 'send_email' decision for this exact draft_id from the approval ledger."""
    log(f"send_draft(draft_id={draft_id[:24]!r}, approval_id={approval_id!r})")
    if not is_approved(approval_id, "send_email", draft_id):
        raise PermissionError("Sending needs a person's approval for this exact draft.")
    return json.dumps(mail.send_draft(draft_id))


if __name__ == "__main__":
    mcp.run(transport="stdio")

In [ ]:
import asyncio
import contextlib
import contextvars
import json
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from langchain_mcp_adapters.tools import load_mcp_tools
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MCP_LOG = Path("mcp_server.log")  # the servers' stderr ([tool called] lines)


def mcp_server(script: str) -> dict:
    return {"script": str(Path(script).resolve())}


@contextlib.asynccontextmanager
async def mcp_session(server: str):
    """Start a local FastMCP server over stdio and open an MCP session with it.

    The server's stderr goes to mcp_server.log: a notebook's own stderr cannot be handed to a subprocess.
    The whole environment is passed on (MCP only forwards a few safe variables by default), so the servers
    receive the Ollama and Azure DevOps settings.
    """
    params = StdioServerParameters(
        command=sys.executable, args=[MCP_SERVERS[server]["script"]], cwd=str(Path.cwd()), env=dict(os.environ)
    )
    with open(MCP_LOG, "a", encoding="utf-8") as errlog:
        async with stdio_client(params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session


def run_async(coro):
    """Run a coroutine from a notebook cell and print the servers' tool log lines.

    It uses a fresh event loop in a worker thread, so it also works in Jupyter on Windows,
    where the notebook's own loop cannot start subprocesses.
    """
    def _run():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        try:
            return loop.run_until_complete(coro)
        finally:
            loop.close()

    log_start = MCP_LOG.stat().st_size if MCP_LOG.exists() else 0
    context = contextvars.copy_context()  # keeps MCP tool calls inside the current LangSmith trace
    with ThreadPoolExecutor(max_workers=1) as pool:
        result = pool.submit(context.run, _run).result()

    if MCP_LOG.exists():
        with open(MCP_LOG, encoding="utf-8") as log:
            log.seek(log_start)
            for line in log.read().splitlines():
                if line.startswith("[") and "tool" in line.split("]")[0]:
                    print(line)
    return result


def _to_python(result):
    if isinstance(result, list):  # recent adapters return a list of content blocks
        result = "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in result)
    try:
        return json.loads(result)
    except (TypeError, json.JSONDecodeError):
        return result


def call_mcp_tools(server: str, calls: list) -> list:
    """Open one MCP session and run several (tool_name, arguments) calls in it."""
    async def _calls():
        async with mcp_session(server) as session:
            tools = {t.name: t for t in await load_mcp_tools(session)}
            results = []
            for name, args in calls:
                try:
                    results.append(_to_python(await tools[name].ainvoke(args)))
                except Exception as error:
                    print(f"[mcp error] {server}.{name}: {error}")
                    results.append({"error": str(error)})
            return results

    return run_async(_calls())


def call_mcp_tool(server: str, tool_name: str, **arguments):
    return call_mcp_tools(server, [(tool_name, arguments)])[0]


def list_mcp_tools(server: str) -> None:
    """tools/list: print the tools a server exposes, with their descriptions and inputs."""
    async def _list():
        async with mcp_session(server) as session:
            return (await session.list_tools()).tools

    print(f"========== TOOLS DISCOVERED ON '{server}' ==========\n")
    for tool in run_async(_list()):
        print(f"Name: {tool.name}")
        print(f"Description: {' '.join(tool.description.split())}")
        print(f"Inputs: {list(tool.inputSchema.get('properties', {}))}\n")
MCP_SERVERS = {"email": mcp_server("email_server.py")}

list_mcp_tools("email")

## Intake: Read a Support Email

The server lists the support emails. The walkthrough continues with the CR-2026-001 request (`MSG-1001` in
the local mailbox); with Outlook connected you choose the email (auto mode takes the newest). The last call
tries to read the confidential HR email, which the server must refuse.

In [ ]:
inbox = call_mcp_tool("email", "list_support_emails", max_emails=10)
show("SUPPORT EMAILS", inbox)
if not inbox:
    raise RuntimeError("No support emails found: check the mailbox folder and the support filter in .env.")

if mail.live:
    for number, item in enumerate(inbox):
        print(f"{number}: {item['subject']}  ({item['from']})")
    request_message_id = inbox[int(input("Email number: ") or 0) if HUMAN_MODE == "interactive" else 0]["message_id"]
else:
    request_message_id = "MSG-1001"

request_email = call_mcp_tool("email", "get_email", message_id=request_message_id)
request_text = request_email["body"]
customer_address = request_email["from_address"]
print(f"\nRequest from {customer_address}:\n\n{request_text}")

if not mail.live:
    print("\n========== Trying to read the confidential email ==========")
    print(call_mcp_tool("email", "get_email", message_id="MSG-1003"))

## Attachments

`save_attachments` downloads the real file attachments of a support email to `attachments/`. Inline images,
such as the logo in an email signature, are skipped. Notebook 04 uploads these files to the Azure DevOps
work item.

In [ ]:
# local mailbox: MSG-1002 shows the inline logo being skipped; Outlook: the first email with files
email_with_files = next((item for item in inbox if item["attachments"]
                         and (mail.live or item["message_id"] == "MSG-1002")), None)
if email_with_files:
    files = call_mcp_tool("email", "save_attachments", message_id=email_with_files["message_id"])
    show(f"ATTACHMENTS OF '{email_with_files['subject']}'", files)
else:
    print("None of the listed support emails has file attachments.")

## Gaps and Retrieved Guidance

The Communication Agent works from the gaps found by the Gap and Retrieval Agent (notebook 01) and from the
template and communication standard stored in the knowledge base (notebook 02). They are included here so
that this notebook can run on its own.

In [ ]:
# Output of the Gap and Retrieval Agent for CR-2026-001 (see notebook 01)
SAMPLE_GAPS = [
    {"gap_id": "GAP-01", "req_id": "REQ-001", "missing_information": "Approval amount and currency",
     "clarification_question": "Above which invoice amount (and in which currency) should an invoice need approval? Is it the amount before or after tax?",
     "assumption_if_unanswered": "Invoice total including tax, in the company's accounting currency."},
    {"gap_id": "GAP-02", "req_id": "REQ-001", "missing_information": "Legal entities in scope",
     "clarification_question": "Which of your companies (legal entities) should use the new approval and blocking rules?",
     "assumption_if_unanswered": "Only the legal entities named by the customer will be configured."},
    {"gap_id": "GAP-03", "req_id": "REQ-001", "missing_information": "Approver and escalation",
     "clarification_question": "Who exactly approves (the finance manager of each company?), and what should happen if they do not respond in time?",
     "assumption_if_unanswered": "The Finance Manager role of each legal entity, no escalation."},
    {"gap_id": "GAP-04", "req_id": "REQ-002", "missing_information": "Meaning of 'blocked'",
     "clarification_question": "When a vendor has no bank details, should we stop only payments, or also the entry of new invoices?",
     "assumption_if_unanswered": "Payments are blocked; invoices can still be entered."},
    {"gap_id": "GAP-05", "req_id": "REQ-002", "missing_information": "Existing vendors",
     "clarification_question": "Should the blocking also apply to existing vendors that are missing bank details today?",
     "assumption_if_unanswered": "Existing and new vendors are included."},
    {"gap_id": "GAP-06", "req_id": "REQ-001", "missing_information": "Go-live date",
     "clarification_question": "By which date does this need to be live (which quarter close)?",
     "assumption_if_unanswered": "The next quarter close."},
]
# Retrieved from the knowledge MCP server in notebook 02 / the final notebook
CLARIFY_TEMPLATE = """Subject: Clarification needed - {request title}

Dear {customer name},

Thank you for your request regarding {one-sentence summary}. To make sure we deliver exactly what you need,
could you please help us with the following questions:

{numbered questions}

Could you please reply by {reply date}? Once we have your answers, we will confirm the final scope before
starting the work.

Kind regards,
{consultant name}
Dynamics 365 F&O Delivery Team"""

COMMUNICATION_STANDARD = (
    "Clarification emails are short, polite and written in business language without technical jargon. "
    "Questions are numbered, one topic per question, with at most eight questions per email. Each email restates "
    "the request in one sentence, asks for a reply date, and never promises a delivery date or solution before the "
    "requirement is approved. Emails are saved as drafts and sent only after a person reviews them."
)
REPLY_BY = (date.today() + timedelta(days=4)).strftime("%d %B %Y")
print("Reply requested by:", REPLY_BY)

## Communication Agent

The agent returns a `ClarificationEmail` object (subject, greeting, one-sentence introduction, numbered
questions, reply date, closing). The final text is rendered in Python so the numbering and the signature are
always consistent.

In [ ]:
import re


@traceable(name="Communication Agent")
def draft_clarification_email(request: str, gaps: list, feedback: str = "") -> ClarificationEmail:
    questions = "\n".join(f"{g['gap_id']}: {g['clarification_question']}" for g in gaps)
    user_prompt = (
        f"Company template:\n{CLARIFY_TEMPLATE}\n\nCommunication standard:\n{COMMUNICATION_STANDARD}\n\n"
        f"Customer request:\n{request}\n\nOpen questions (keep their order):\n{questions}\n\n"
        f"Reply date: {REPLY_BY}\nConsultant name: {APPROVER}"
    )
    if feedback:
        user_prompt += f"\n\nReviewer feedback on the previous draft: {feedback}"
    return ask_structured(ClarificationEmail, COMMUNICATION_SYSTEM, user_prompt)


def render_email(email: ClarificationEmail) -> str:
    questions = "\n".join(  # the numbering is added here, so any number the model wrote is removed
        f"{number}. {re.sub(r'^\s*\d+[.)]\s*', '', question)}"
        for number, question in enumerate(email.questions, start=1)
    )
    return (f"{email.greeting}\n\n{email.introduction}\n\n{questions}\n\n"
            f"Could you please reply by {email.reply_by}?\n\n{email.closing}\n{APPROVER}\n"
            "Dynamics 365 F&O Delivery Team")


email = draft_clarification_email(request_text, SAMPLE_GAPS)
print("Subject:", email.subject, "\n")
print(render_email(email))

## Guardrail Test: Saving or Sending Without Approval

An agent (or a prompt injection inside an email) might try to save or send a reply directly. Both tools check
the ledger and refuse because no person approved it.

In [ ]:
print(call_mcp_tool("email", "save_reply_draft", message_id=request_message_id, subject=email.subject,
                    body=render_email(email), approval_id="APR-FAKE0001"))
print(call_mcp_tool("email", "send_draft", draft_id="DRAFT-999", approval_id="APR-FAKE0002"))

## Human Approval, Save as Draft, Then Send

Two separate decisions:

1. **The email text**: approve, revise (with comments that go back to the Communication Agent) or reject.
   An approved email is saved as a draft in the mailbox.
2. **Sending that exact draft**: approve or reject. A rejected draft stays in the Drafts folder.

In auto mode both are approved for the local mailbox. With Outlook connected, auto mode never sends a real
email: set `HUMAN_MODE = "interactive"` above to make the decisions yourself.

In [ ]:
for attempt in range(3):
    review = human_decision(f"Clarification email (draft {attempt + 1})", f"Subject: {email.subject}\n\n{render_email(email)}")
    record = approval_ledger.record_decision(
        "customer_email", request_message_id, APPROVER,
        {"approve": "approved", "revise": "revise", "reject": "rejected"}[review["decision"]], review["comments"],
    )
    if review["decision"] != "revise":
        break
    email = draft_clarification_email(request_text, SAMPLE_GAPS, feedback=review["comments"])

if record["decision"] == "approved":
    draft = call_mcp_tool("email", "save_reply_draft", message_id=request_message_id, subject=email.subject,
                          body=render_email(email), approval_id=record["approval_id"])
    if "error" in draft:
        raise RuntimeError(draft["error"])
    show("SAVED DRAFT", draft)

    send_review = human_decision(f"Send this draft to {', '.join(draft['to'] + draft['cc'])}?",
                                 f"Subject: {draft['subject']}", options=("approve", "reject"),
                                 live_action=draft["mode"] == "live")
    send_record = approval_ledger.record_decision(
        "send_email", draft["draft_id"], APPROVER,
        "approved" if send_review["decision"] == "approve" else "rejected", send_review["comments"],
    )
    if send_record["decision"] == "approved":
        show("SENT", call_mcp_tool("email", "send_draft", draft_id=draft["draft_id"], approval_id=send_record["approval_id"]))
    else:
        print("Not sent: the draft stays in the mailbox for a person to send or delete.")
else:
    print("The email was rejected. Nothing was saved.")

show("APPROVAL LEDGER", json.loads(Path("approvals.json").read_text()))

## The Customer Replies

The workflow is paused until the customer answers. In `auto` mode the reply below is used; in `interactive`
mode you can paste a reply yourself.

In [ ]:
# The customer's answer to the clarification email (used when HUMAN_MODE = "auto")
SIMULATED_CUSTOMER_REPLY = """
Hi,

Thanks for the questions. Answers below:
- The threshold is 10,000 USD per invoice, compared in the accounting currency of each company,
  and it applies to the invoice total including tax.
- Only the legal entities USMF and DEMF are in scope for now.
- The approver is the Finance Manager of each legal entity. If they do not act within 2 business
  days, the approval should escalate to the CFO.
- Invoices of 10,000 USD or less can be posted by AP clerks without approval, like today.
- "Blocked" means the vendor is put on payment hold so no payments can be generated; invoices can
  still be registered.
- It applies to existing vendors and to new vendors. A vendor has bank details when it has at least
  one active bank account.
- The auditors need to see who approved each invoice and when.
- AP clerks must not be able to approve invoices they entered themselves.
- Go-live must be before 15 December 2026.

Regards,
Rana
"""
if HUMAN_MODE == "interactive":
    print("Paste the customer's reply and finish with an empty line:")
    customer_reply = "\n".join(iter(input, ""))
else:
    customer_reply = SIMULATED_CUSTOMER_REPLY
print(customer_reply)

## Answer-Mapping Agent

The reply is mapped back to the gap IDs. Answered gaps become **confirmed answers**. Gaps the customer did not
answer fall back to their **assumption**, which stays visibly separate from confirmed facts so that the
specification approval can challenge it.

In [ ]:
gap_list = "\n".join(f"{g['gap_id']}: {g['clarification_question']}" for g in SAMPLE_GAPS)
mapped = ask_structured(CustomerAnswers, ANSWERS_SYSTEM, f"Open gaps:\n{gap_list}\n\nCustomer reply:\n{customer_reply}")

confirmed_answers = {a.gap_id: a.answer for a in mapped.answers}
assumptions = [
    f"{g['gap_id']}: {g['assumption_if_unanswered']}" for g in SAMPLE_GAPS if g["gap_id"] not in confirmed_answers
]

show("CONFIRMED ANSWERS", confirmed_answers)
show("ASSUMPTIONS (UNANSWERED GAPS)", assumptions)

## Observation

The request entered through the support filter: the server listed only support emails and refused to open the
confidential HR email. Attachments were downloaded without the inline signature logo. The Communication Agent
produced a short email with numbered questions in business language, following the company template, and
without promising a date or a solution.

The guardrails worked at the tool level: `save_reply_draft` and `send_draft` refused to act without a real
approval in the ledger. After the human approved the text, the email was saved as a draft; sending needed a
second approval for that exact draft, and every decision was recorded with an approval ID, approver and
timestamp. With Outlook connected, the same tools work on the real mailbox through Microsoft Graph, and auto
mode never sends a real email.

The customer's reply was mapped back to the gap IDs. Answered gaps became confirmed answers. The one gap the
reply might not cover keeps its assumption and is shown separately, so the specification approval in the
final notebook can decide whether it is acceptable. The confirmed answers become immutable facts from that point on.